# Create RDF Data and Upload to LINDAS

This notebook contains code to create RDF data and upload it to the LINDAS triple store.

## Read Party JSON

In [ ]:
import json

# read JSON file to dict
with open('../data/output/parties.json', 'r') as file:
    parties = json.load(file)

## Convert Party JSON to RDF

### URIs and Naming

In [ ]:
# identities and versions
PARTY_IDENTITY = "https://politics.ld.admin.ch/political-party"
PARTY_VERSION = "https://politics.ld.admin.ch/political-party-version"

# shared dimensions
PARTY_SHARED_DIMENSION_IDENTITIES = "https://politics.ld.admin.ch/political-party"
PARTY_SHARED_DIMENSION_VERSIONS = "https://politics.ld.admin.ch/political-party-version"

# versioned identity set
PARTY_VERSIONED_IDENTITY_SET = "https://politics.ld.admin.ch/.well-known/void/dataset/party-id/versioned-identity-set"

# predicates namespace
PARTY_PREFIX = "pol"
PARTY_VOCAB = "https://politics.ld.admin.ch/vocabulary/political-party/"

PARTY_PREDICATE_STARTED_BY = "startedBy"
PARTY_PREDICATE_STOPPED_BY = "stoppedBy"

# dimensions namespace
PARTY_DIMENSIONS = "https://politics.ld.admin.ch/vocabulary/political-party"

# dimensions
PARTY_STARTING = f"{PARTY_DIMENSIONS}/starting-class"
PARTY_STOPPING = f"{PARTY_DIMENSIONS}/stopping-class"

In [ ]:
event_mapping = {
    "founding": "Founding",
    "dissolution": "Dissolution",
    "name-change_start": "NameChangeStart",
    "name-change_stop": "NameChangeStop",
    "successor-of": "SuccessorOf",
    "split-from": "SplitFrom",
    "accessed": "Accessed",
    "child-of_start": "ChildOfStart",
    "child-of_stop": "ChildOfStop",
    "observer-in_start": "ObserverInStart",
    "observer-in_stop": "ObserverInStop",
    "affiliated-with_start": "AffiliatedWithStart",
    "affiliated-with_stop": "AffiliatedWithStop"
}

### Transformation to RDF

In [ ]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace

# create graph without binding namespaces to avoid automatic prefixing
g = Graph(bind_namespaces="none")

schema = Namespace("http://schema.org/")
g.bind("schema", schema)
xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
g.bind("xsd", xsd)
owl = Namespace("http://www.w3.org/2002/07/owl#")
g.bind("owl", owl)
cube = Namespace("https://cube.link/")
g.bind("cube", cube)
vl = Namespace("https://version.link/")
g.bind("vl", vl)
pol = Namespace(PARTY_VOCAB)
g.bind(PARTY_PREFIX, pol)

# create shared dimensions and versioned identity set
g.add((URIRef(PARTY_SHARED_DIMENSION_IDENTITIES), RDF.type, schema.DefinedTermSet))
g.add((URIRef(PARTY_SHARED_DIMENSION_IDENTITIES), RDF.type, cube.SharedDimension))
g.add((URIRef(PARTY_SHARED_DIMENSION_IDENTITIES), schema.name, Literal("Party identities", lang="en")))
g.add((URIRef(PARTY_SHARED_DIMENSION_IDENTITIES), schema.description, Literal("This shared dimension contains all party identities.", lang="en")))

g.add((URIRef(PARTY_SHARED_DIMENSION_VERSIONS), RDF.type, vl.SharedDimension))
g.add((URIRef(PARTY_SHARED_DIMENSION_VERSIONS), RDF.type, cube.SharedDimension))
g.add((URIRef(PARTY_SHARED_DIMENSION_VERSIONS), schema.name, Literal("Party versions", lang="en")))
g.add((URIRef(PARTY_SHARED_DIMENSION_VERSIONS), schema.description, Literal("This shared dimension contains all party versions.", lang="en")))

g.add((URIRef(PARTY_VERSIONED_IDENTITY_SET), RDF.type, vl.VersionedIdentitySet))

# create identities and versions for each party
for party_id, party in parties.items():
    
    # create identity
    identity_uri = URIRef(f"{PARTY_IDENTITY}/{party_id}")
    g.add((identity_uri, RDF.type, vl.Identity))
    g.add((identity_uri, RDF.type, schema.PoliticalParty))
    if not pd.isna(party["dissolution_date"]):
        g.add((identity_uri, RDF.type, vl.Deprecated))

    # add identifier
    g.add((identity_uri, schema.identifier, Literal(str(party_id))))

    # add BFS identifier to the identity
    bfs_code = next((code["code"] for code in party["party_codes"] if code["type_label"] == "bfs-code"), None)
    if bfs_code:
        g.add((identity_uri, pol.bfsIdentifier, Literal(str(bfs_code))))

    # add to shared dimension
    g.add((identity_uri, schema.inDefinedTermSet, URIRef(PARTY_SHARED_DIMENSION_IDENTITIES)))
    g.add((URIRef(PARTY_SHARED_DIMENSION_IDENTITIES), schema.hasDefinedTerm, identity_uri))

    # add to versioned identity set
    g.add((identity_uri, vl.inVersionedIdentitySet, URIRef(PARTY_VERSIONED_IDENTITY_SET)))
    g.add((URIRef(PARTY_VERSIONED_IDENTITY_SET), schema.hasPart, identity_uri)) # there is no inverse of vl.inVersionedIdentitySet

    # add area
    if not pd.isna(party["area"]["uri"]):
        g.add((identity_uri, schema.areaServed, URIRef(party["area"]["uri"])))

    # link to the last version
    g.add((identity_uri, vl.version, URIRef(f"{PARTY_VERSION}/{party_id}_{party['version_ids'][-1]}")))

    # add wikidata identifier if it exists
    wikidata_code = next((code["code"] for code in party["party_codes"] if code["type_label"] == "wikidata-code"), None)
    if wikidata_code:
        g.add((identity_uri, owl.sameAs, URIRef(f"https://www.wikidata.org/wiki/{wikidata_code}")))

    # add color
    if not pd.isna(party["color"]):
        g.add((identity_uri, schema.color, Literal(party["color"])))

    # add founding date to the identity
    g.add((identity_uri, schema.validFrom, Literal(party["founding_date"], datatype=xsd.date)))

    # create versions
    for version_id, version in party["versions"].items():
                    
        version_uri = URIRef(f"{PARTY_VERSION}/{party_id}_{version_id}")
        g.add((version_uri, RDF.type, vl.Version))
        g.add((version_uri, RDF.type, schema.PoliticalParty))
        g.add((version_uri, vl.identity, identity_uri))

        # add identifier to the version
        g.add((version_uri, schema.identifier, Literal(str(f"{party_id}_{version_id}"))))

        # add identifier of identity to the version
        g.add((version_uri, vl.identityIdentifier, Literal(str(party_id))))

        # add to shared dimension
        g.add((version_uri, schema.inDefinedTermSet, URIRef(PARTY_SHARED_DIMENSION_VERSIONS)))
        g.add((URIRef(PARTY_SHARED_DIMENSION_VERSIONS), schema.hasDefinedTerm, version_uri))

        # add to versioned identity set
        g.add((version_uri, vl.inVersionedIdentitySet, URIRef(PARTY_VERSIONED_IDENTITY_SET)))
        g.add((URIRef(PARTY_VERSIONED_IDENTITY_SET), schema.hasPart, version_uri)) # there is no inverse of vl.inVersionedIdentitySet

        # add area
        if not pd.isna(party["area"]["uri"]):
            g.add((version_uri, schema.areaServed, URIRef(party["area"]["uri"])))

        # add dates
        g.add((version_uri, schema.validFrom, Literal(version["valid_from"], datatype=xsd.date)))
        if not pd.isna(version["valid_through"]):
            # set validThrough one day before
            valid_through = pd.to_datetime(version["valid_through"]) - pd.Timedelta(days=1)
            valid_through = str(valid_through.date())  # convert to date object
            g.add((version_uri, schema.validThrough, Literal(valid_through, datatype=xsd.date)))
            # if last version
            if version_id == party['version_ids'][-1]:
                g.add((identity_uri, schema.validThrough, Literal(valid_through, datatype=xsd.date)))

        # add name
        name_entry = version["name"]
        if not pd.isna(name_entry):
            for lang, name in name_entry.items():
                g.add((version_uri, schema.name, Literal(name, lang=lang)))
                if version_id == party['version_ids'][-1]:  # if this is the last version, also add the name to the identity
                    g.add((identity_uri, schema.name, Literal(name, lang=lang)))

        # add abbreviation
        abbr_entry = version["abbr"]
        if not pd.isna(abbr_entry):
            for lang, abbr in abbr_entry.items():
                g.add((version_uri, schema.alternateName, Literal(abbr, lang=lang)))
                if version_id == party['version_ids'][-1]:  # if this is the last version, also add the abbreviation to the identity
                    g.add((identity_uri, schema.alternateName, Literal(abbr, lang=lang)))

        # add relationships
        for relationship in version["relationships"]:
            # convert relationship type from hyphenated to camelCase for RDF property
            first, second = relationship["type_label"].split("-")
            relationship_type = first + second.capitalize()  # e.g. child-of becomes childOf
            g.add((version_uri, pol[f"{relationship_type}"], URIRef(f"{PARTY_IDENTITY}/{relationship['party_to']}")))
            if relationship_type == "childOf": # add also schema:isPartOf for childOf relationships
                g.add((version_uri, schema.isPartOf, URIRef(f"{PARTY_IDENTITY}/{relationship['party_to']}")))
            if version_id == party['version_ids'][-1]:  # if this is the last version, also add the relationship to the identity
                g.add((identity_uri, pol[f"{relationship_type}"], URIRef(f"{PARTY_IDENTITY}/{relationship['party_to']}")))

        # # add starting classes
        # for starting_event in version["starting_events"]:
        #     g.add((version_uri, pol[f"{PARTY_PREDICATE_STARTED_BY}"], URIRef(f"{PARTY_STARTING}/{event_mapping.get(starting_event['event_type'])}")))

        # # add stopping classes
        # for stopping_event in version["stopping_events"]:
        #     g.add((version_uri, pol[f"{PARTY_PREDICATE_STOPPED_BY}"], URIRef(f"{PARTY_STOPPING}/{event_mapping.get(stopping_event['event_type'])}")))

        # link to successors and predecessors
        for successor in version["successors"]:
            g.add((version_uri, vl.successor, URIRef(f"{PARTY_VERSION}/{successor}")))
            g.add((URIRef(f"{PARTY_VERSION}/{successor}"), vl.predecessor, version_uri))

g.serialize(destination="../data/output/parties.ttl", format="turtle")

## Load Environment Variables

In [1]:
from dotenv import load_dotenv
import os

ENV = "INT"

# Load the env file
load_dotenv(f"../.env.{ENV.lower()}")

print(os.getenv('LINDAS_USERNAME'))
print(os.getenv('GRAPH'))

lindas-fch-political-party
https://lindas.admin.ch/fch/political-party


## Upload RDF Data

In [2]:
import requests
import os
from pathlib import Path
from requests.auth import HTTPBasicAuth

# Read ENV variables
USERNAME = os.getenv('LINDAS_USERNAME')
PASSWORD = os.getenv('LINDAS_PASSWORD')
GRAPH = os.getenv("GRAPH")
ENDPOINT = os.getenv("ENDPOINT")

params = {
    'graph': GRAPH
}

# Upload the new data

# Define the headers
headers = {
    'Content-Type': 'text/turtle',
}

folder = Path("../data/output")
files = ttl_files = [f.name for f in folder.glob("*.ttl")]

for i, file in enumerate(files):
    if not os.path.exists(f"../data/output/{file}"):
        print(f"File {file} does not exist.")
        continue
    print(f"Uploading {file} to graph {GRAPH}")

    with open(f"../data/output/{file}", 'r') as f:
        # Make the PUT request with the file data, headers, and authentication
        
        if i == 0:
            # Use PUT for the first file to delete existing data
            response = requests.put(ENDPOINT, params=params, headers=headers, data=f, auth=HTTPBasicAuth(USERNAME, PASSWORD))
        else:
            # Use POST for subsequent files to append data
            response = requests.post(ENDPOINT, params=params, headers=headers, data=f, auth=HTTPBasicAuth(USERNAME, PASSWORD))

    if response.status_code == 204:
        print(f"Upload succeeded for {file} with status code {response.status_code}")
    else:
        print(f"Upload failed for {file} with status code {response.status_code}")
        print(response.text)

Uploading parties.ttl to graph https://lindas.admin.ch/fch/political-party
Upload succeeded for parties.ttl with status code 204
